<a href="https://colab.research.google.com/github/JSJeong-me/GPT-Insights/blob/main/10%20Image%20Generation/video_generation_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Veo 모델 기반 멀티모달 비디오 생성 및 시네마틱 연출 실습

**💡 사전 필수 준비 (API Key 설정)**
1. Colab 왼쪽 사이드바에서 **'열쇠 아이콘'(보안 비밀)**을 클릭합니다.
2. 새 비밀 추가: 이름에 `GEMINI_API_KEY` 입력, 값에 API 키 붙여넣기
3. **'노트북 액세스' 토글 버튼을 켜서 활성화**합니다.

## 제1장. 강의 개요 및 비디오 생성 환경 구축

In [10]:
# 1-1. 최신 SDK 설치
!pip install -q -U google-genai pillow requests

import time
import requests
import os
from google.colab import userdata
import google.genai as genai
from google.genai import types
from IPython.display import Image, Markdown, display, Video
from PIL import Image as PILImage

# 1-2. Client 초기화
try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)
    print("✅ Gemini API Client 초기화 성공!")
except userdata.SecretNotFoundError:
    print("❌ 에러: 'GEMINI_API_KEY'가 설정되지 않았습니다.")

# 모델 ID 설정: 오류 해결을 위해 적절한 모델 ID로 변경
IMAGE_MODEL_ID = "models/gemini-1.5-pro"
TEXT_MODEL_ID = "models/gemini-1.5-pro"
VIDEO_MODEL_ID = "models/veo-2.0-generate-001"

# (유틸리티) 비디오 출력 함수
def show_video(video_obj, filename):
    video_bytes = getattr(video_obj, 'video_bytes', None)
    uri = getattr(video_obj, 'uri', None)

    if video_bytes:
        with open(filename, "wb") as f:
            f.write(video_bytes)
        display(Video(filename, embed=True, width=600))
    elif uri:
        print(f"⏳ URI에서 비디오 다운로드 중...")
        response = requests.get(f"{uri}&key={api_key}")
        if response.status_code == 200:
            with open(filename, "wb") as f:
                f.write(response.content)
            display(Video(filename, embed=True, width=600))
        else:
            print(f"❌ 다운로드 실패 (상태 코드: {response.status_code})")
    else:
        print("⚠️ 비디오 데이터를 찾을 수 없습니다.")

✅ Gemini API Client 초기화 성공!


## 제2장. 텍스트 기반 기본 비디오 생성 (Text-to-Video)

In [2]:
print("⏳ [1/4] 기초 영상을 렌더링 중입니다...")

operation = client.models.generate_videos(
    model=VIDEO_MODEL_ID,
    prompt="A person explaining the Pythagorean theorem.",
    config=types.GenerateVideosConfig(
        aspect_ratio="16:9",
        number_of_videos=1,
        duration_seconds=5,
        resolution="720p",
        person_generation="allow_adult"
    ),
)

while not operation.done:
    print(".", end="")
    time.sleep(15)
    operation = client.operations.get(operation)

res = operation.response if operation.response else operation.result

if res and hasattr(res, 'generated_videos') and res.generated_videos:
    print("\n✅ 기초 비디오 렌더링 완료!")
    generated_video = res.generated_videos[0].video
    show_video(generated_video, "basic_video.mp4")
else:
    print("\n❌ 비디오 생성에 실패했습니다.")

⏳ [1/4] 기초 영상을 렌더링 중입니다...
...
✅ 기초 비디오 렌더링 완료!
⏳ URI에서 비디오 다운로드 중...


## 사용 가능한 비디오 생성 모델 확인

In [3]:
print("⏳ 사용 가능한 모델을 불러오는 중...")

# API에서 사용 가능한 모든 모델 목록을 가져옵니다.
models = client.models.list()

# 비디오 생성에 사용될 수 있는 모델만 필터링합니다.
video_models = []
for model in models:
    if hasattr(model, 'supported_generation_methods'):
        if 'generateVideos' in model.supported_generation_methods:
            video_models.append(model)

if video_models:
    print("✅ 다음은 비디오 생성을 지원하는 모델 목록입니다:")
    for model in video_models:
        print(f"- {model.name} (버전: {model.version})")
else:
    print("❌ 현재 비디오 생성을 지원하는 모델을 찾을 수 없습니다.")

# 현재 사용 중인 VIDEO_MODEL_ID가 목록에 있는지 확인합니다.
current_model_found = any(model.name == VIDEO_MODEL_ID for model in video_models)
if not current_model_found:
    print(f"\n⚠️ 현재 설정된 VIDEO_MODEL_ID('{VIDEO_MODEL_ID}')는 사용 가능한 목록에 없습니다.")
    print("위 목록에서 적절한 모델 ID를 선택하여 'VIDEO_MODEL_ID' 변수를 업데이트해주세요.")

print("\n이 목록에서 `VIDEO_MODEL_ID` 변수를 업데이트하여 사용하세요.")

⏳ 사용 가능한 모델을 불러오는 중...
❌ 현재 비디오 생성을 지원하는 모델을 찾을 수 없습니다.

⚠️ 현재 설정된 VIDEO_MODEL_ID('models/veo-2.0-generate-001')는 사용 가능한 목록에 없습니다.
위 목록에서 적절한 모델 ID를 선택하여 'VIDEO_MODEL_ID' 변수를 업데이트해주세요.

이 목록에서 `VIDEO_MODEL_ID` 변수를 업데이트하여 사용하세요.


In [4]:
import google.genai as genai
from google.colab import userdata

# 클라이언트 재초기화
try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)

    print("🔍 검색 가능한 모든 모델 목록:")
    models = client.models.list()
    for model in models:
        # 모델 객체에 직접 속성이 없을 경우를 대비해 hasattr 체크
        methods = getattr(model, 'supported_generation_methods', [])
        print(f"ID: {model.name}, 지원 기능: {methods}")
except Exception as e:
    print(f"❌ 오류 발생: {e}")

🔍 검색 가능한 모든 모델 목록:
ID: models/gemini-2.5-flash, 지원 기능: []
ID: models/gemini-2.5-pro, 지원 기능: []
ID: models/gemini-2.0-flash, 지원 기능: []
ID: models/gemini-2.0-flash-001, 지원 기능: []
ID: models/gemini-2.0-flash-lite-001, 지원 기능: []
ID: models/gemini-2.0-flash-lite, 지원 기능: []
ID: models/gemini-2.5-flash-preview-tts, 지원 기능: []
ID: models/gemini-2.5-pro-preview-tts, 지원 기능: []
ID: models/gemma-4-26b-a4b-it, 지원 기능: []
ID: models/gemma-4-31b-it, 지원 기능: []
ID: models/gemini-flash-latest, 지원 기능: []
ID: models/gemini-flash-lite-latest, 지원 기능: []
ID: models/gemini-pro-latest, 지원 기능: []
ID: models/gemini-2.5-flash-lite, 지원 기능: []
ID: models/gemini-2.5-flash-image, 지원 기능: []
ID: models/gemini-3-pro-preview, 지원 기능: []
ID: models/gemini-3-flash-preview, 지원 기능: []
ID: models/gemini-3.1-pro-preview, 지원 기능: []
ID: models/gemini-3.1-pro-preview-customtools, 지원 기능: []
ID: models/gemini-3.1-flash-lite-preview, 지원 기능: []
ID: models/gemini-3.1-flash-lite, 지원 기능: []
ID: models/gemini-3-pro-image-preview, 지원 기능: []


## 제3장. 연속성 유지를 위한 베이스 프레임 설계 (Image-to-Video 준비)

In [5]:
print("⏳ [2/4] 베이스 이미지 렌더링 중...")

# 3-1. 슬라이드 더미 이미지 생성
slide_img = PILImage.new('RGB', (800, 450), color = '#eef2f5')
slide_img.save("slide-image.png")

with open("slide-image.png", "rb") as f:
    image_bytes = f.read()

# 3-2. 슬라이드 이미지를 참조하여 베이스 프레임 생성
try:
    response = client.models.generate_content(
        model=IMAGE_MODEL_ID,
        contents=[
            types.Part.from_bytes(data=image_bytes, mime_type="image/png"),
            "Generate a high-quality photorealistic image of a professor in a classroom full of students, standing in front of a presentation slide."
        ],
        config=types.GenerateContentConfig(
            response_modalities=['IMAGE'],
            image_config=types.ImageConfig(aspect_ratio="16:9"),
        ),
    )

    img_found = False
    for part in response.candidates[0].content.parts:
        if part.inline_data:
            img_data = part.inline_data.data
            display(Image(data=img_data, width=500))
            with open("video-image.png", "wb") as f:
                f.write(img_data)
            img_found = True
            print("✅ 영상의 베이스 프레임(video-image.png) 저장 완료!")

    if not img_found:
        print("⚠️ 모델 응답에 이미지 데이터가 포함되어 있지 않습니다.")
except Exception as e:
    print(f"❌ 이미지 생성 오류: {e}")

⏳ [2/4] 베이스 이미지 렌더링 중...
❌ 이미지 생성 오류: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Model does not support the requested response modalities: image', 'status': 'INVALID_ARGUMENT'}}


## 제4장. 시네마틱 프롬프트 엔지니어링 (Cinematic Prompt Enhancement)

In [6]:
# 4-1. 영상 연출 변수 정의
subject = "a professor"
action = "giving a lecture on the Pythagorean theorem"
scene = "in a classroom, presenting in front of students"
style = "photorealistic"
camera_angle = "eye-level shot"
camera_movement = "zoom in"
sound_effects = "clicking of computer keys"
dialogue = "When you square a side length, like a or b, you are literally calculating the area of a square built perfectly off that side."

keywords = [subject, action, scene, style, camera_angle, camera_movement, sound_effects, dialogue]

# 4-2. LLM을 통한 프롬프트 합성
gemini_prompt = f"""
Your task is to expand the following keywords into a single, high-fidelity,
descriptive prompt for video generation. Every single keyword MUST be
included. Synthesize them into a single, cohesive, and cinematic
instruction. Output ONLY the final prompt string.

Mandatory Keywords: {", ".join(keywords)}
"""

try:
    # TEXT_MODEL_ID(gemini-1.5-flash)를 사용하여 비디오용 프롬프트 생성
    response = client.models.generate_content(
        model=TEXT_MODEL_ID,
        contents=gemini_prompt,
    )
    video_prompt = response.text
    display(Markdown(f"**🎬 완성된 시네마틱 촬영 지시서:**\n> {video_prompt}"))
except Exception as e:
    print(f"❌ 프롬프트 생성 중 오류 발생: {e}")
    # 오류 시 기본 프롬프트 설정하여 NameError 방지
    video_prompt = f"{subject} {action} {scene} in {style} style, {camera_angle}, {camera_movement}."
    print("⚠️ 기본 프롬프트를 사용합니다.")

❌ 프롬프트 생성 중 오류 발생: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}
⚠️ 기본 프롬프트를 사용합니다.


## 제5장. 고품질 최종 영상 렌더링 및 실무 응용 (Image-to-Video)

In [7]:
import os

print("⏳ [4/4] 최종 시네마틱 비디오 렌더링 중입니다...")

if not os.path.exists("video-image.png"):
    print("❌ 오류: 'video-image.png' 파일이 없습니다. 3장 셀(zw3LZGpgzRJ6)을 다시 실행해 주세요.")
else:
    try:
        operation = client.models.generate_videos(
            model=VIDEO_MODEL_ID,
            prompt=video_prompt,
            image=types.Image.from_file(location="video-image.png"),
            config=types.GenerateVideosConfig(
                aspect_ratio="16:9",
                number_of_videos=1,
                duration_seconds=5,
                resolution="720p",
                person_generation="allow_adult"
            ),
        )

        while not operation.done:
            print(".", end="")
            time.sleep(15)
            operation = client.operations.get(operation)

        res = operation.response if operation.response else operation.result

        if res and hasattr(res, 'generated_videos') and res.generated_videos:
            print("\n✅ 최종 비디오 렌더링 완료!")
            generated_video = res.generated_videos[0].video
            show_video(generated_video, "final_cinematic_video.mp4")
            display(Markdown("**🎉 축하합니다! `final_cinematic_video.mp4` 파일이 생성되었습니다.**"))
        else:
            print("\n❌ 비디오 생성 실패.")
    except Exception as e:
        print(f"\n❌ 오류 발생: {e}")

⏳ [4/4] 최종 시네마틱 비디오 렌더링 중입니다...
❌ 오류: 'video-image.png' 파일이 없습니다. 3장 셀(zw3LZGpgzRJ6)을 다시 실행해 주세요.
